In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

import os

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

/home/kejh66/anaconda3/envs/lad/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/kejh66/anaconda3/envs/lad/lib/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

2025-11-06 15:19:29.029729: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.86it/s]


In [3]:
log_datasets = {
    "tcp_recv": "dataset/user-defined_dataset/android_tcp_recv.txt",
    "ssl_support": "dataset/user-defined_dataset/apache_ssl_support.txt",
    "resourcemgmt_fail": "dataset/user-defined_dataset/hpc_resourcemgmt_fail.txt",
    "hw_unavailable": "dataset/user-defined_dataset/hpc_hw_unavailable.txt"
}

labels_datasets = {
    "tcp_recv": "dataset/user-defined_dataset/android_tcp_recv_labeling.txt",
    "ssl_support": "dataset/user-defined_dataset/apache_ssl_support_labeling.txt",
    "resourcemgmt_fail": "dataset/user-defined_dataset/hpc_resourcemgmt_fail_labeling.txt",
    "hw_unavailable": "dataset/user-defined_dataset/hpc_hw_unavailable_labeling.txt"
}

user_anomaly_definition = {
    "tcp_recv": "An anomaly occurs when a network socket fails to read data, indicating a communication error or network failure.",
    "ssl_support": "An anomaly occurs when SSL configuration is missing or invalid in a secured setup.",
    "resourcemgmt_fail": "An anomaly occurs when the system fails to configure the resource management subsystem.",
    "hw_unavailable": "An anomaly occurs when a hardware component becomes unavailable, indicating device failure or disconnection."
}

In [4]:
def read_log_file(dataset_name, dataset_dict):
    if dataset_name not in dataset_dict:
        raise ValueError(f"Dataset '{dataset_name}' not found. Available: {list(dataset_dict.keys())}")
    
    file_path = dataset_dict[dataset_name]
    
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File does not exist: {file_path}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.rstrip("\n") for line in f if line.strip()]
    
    return lines

In [5]:
def build_messages(log_lines, user_anomaly_definition):
    indexed_logs = "\n".join([f"{i+1}. {line}" for i, line in enumerate(log_lines)])

    prompt_text = f"""
You are an expert system log anomaly detector.

Analyze each log entry and determine whether it represents NORMAL or ABNORMAL system behavior based solely on the following user-defined rule:
{user_anomaly_definition}

If a line is ABNORMAL, provide a short and clear explanation of why it is abnormal, including the main cause or affected component. 
Keep the explanation to one or two short sentences.

Output format (strictly follow this, no explanations beyond what is requested):
1. normal
2. abnormal - explanation
3. normal
4. abnormal - explanation
...

Logs:
{indexed_logs}

Now produce the classifications following the format above.
""".strip()

    messages = [
        {"role": "system", "content": "You are Qwen, a helpful assistant specialized in system log anomaly detection."},
        {"role": "user", "content": prompt_text}
    ]
    return messages


In [6]:
def parse_model_output(model_output):
    lines = model_output.strip().split("\n")
    labels = []
    for line in lines:
        line = line.lower()
        if "abnormal" in line:
            labels.append(1)
        elif "normal" in line:
            labels.append(0)
        else:
            labels.append(-1)
    return labels

In [7]:
def read_ground_truth(dataset_name, dataset_dict):
    path = dataset_dict[dataset_name]
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    
    labels = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line == "":
                continue
            # anomaly=1, normal=0
            labels.append(int(line))
    return labels

In [8]:
def evaluate_predictions(model_output, dataset_name, dataset_dict):
    y_pred = parse_model_output(model_output)
    y_true = read_ground_truth(dataset_name, dataset_dict)

    if len(y_pred) != len(y_true):
        print(f"Warning: prediction length {len(y_pred)} != ground truth length {len(y_true)}")
        min_len = min(len(y_pred), len(y_true))
        y_pred = y_pred[:min_len]
        y_true = y_true[:min_len]

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)

    print(f"=== Evaluation {dataset_name} Results ===")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")

### TCP Recv

In [12]:
selected_dataset = "tcp_recv"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines, user_anomaly_definition[selected_dataset])

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [13]:
print(response)

(1) normal
(2) abnormal - socket read length failure -104
(3) normal
(4) normal
(5) normal
(6) normal
(7) normal
(8) normal
(9) normal
(10) abnormal - socket read length failure -104
(11) normal
(12) normal
(13) normal
(14) normal
(15) normal
(16) abnormal - socket read length failure -1
(17) normal
(18) abnormal - socket read length failure -1
(19) normal
(20) normal
(21) normal
(22) normal
(23) normal
(24) abnormal - socket read length failure -1
(25) abnormal - socket read length failure -1


In [14]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation tcp_recv Results ===
Accuracy:  0.9600
Precision: 0.8333
Recall:    1.0000
F1-score:  0.9091


### SSL Support

In [15]:
selected_dataset = "ssl_support"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines, user_anomaly_definition[selected_dataset])

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [16]:
print(response)

(1) normal
(2) abnormal - SSL support unavailable
(3) normal
(4) normal
(5) normal
(6) abnormal - SSL support unavailable
(7) abnormal - SSL support unavailable
(8) normal
(9) normal
(10) normal
(11) normal
(12) abnormal - SSL support unavailable
(13) abnormal - SSL support unavailable
(14) normal
(15) normal
(16) normal
(17) normal
(18) normal
(19) normal
(20) normal
(21) normal
(22) normal
(23) normal
(24) normal
(25) normal


In [17]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation ssl_support Results ===
Accuracy:  0.9600
Precision: 1.0000
Recall:    0.8333
F1-score:  0.9091


### Resourcement Fail

In [24]:
selected_dataset = "resourcemgmt_fail"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines, user_anomaly_definition[selected_dataset])

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [25]:
print(response)

(1) normal
(2) abnormal - resourcemgmt subsystem configuration failure
(3) normal
(4) abnormal - resourcemgmt subsystem configuration failure
(5) normal
(6) normal
(7) normal
(8) normal
(9) normal
(10) abnormal - resourcemgmt subsystem configuration failure
(11) abnormal - resourcemgmt subsystem configuration failure
(12) abnormal - resourcemgmt subsystem configuration failure
(13) abnormal - resourcemgmt subsystem configuration failure
(14) abnormal - resourcemgmt subsystem configuration failure
(15) abnormal - resourcemgmt subsystem configuration failure
(16) abnormal - resourcemgmt subsystem configuration failure
(17) normal
(18) normal
(19) normal
(20) normal
(21) normal
(22) abnormal - resourcemgmt subsystem configuration failure
(23) abnormal - resourcemgmt subsystem configuration failure
(24) abnormal - resourcemgmt subsystem configuration failure
(25) abnormal - resourcemgmt subsystem configuration failure


In [26]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation resourcemgmt_fail Results ===
Accuracy:  0.9200
Precision: 0.8462
Recall:    1.0000
F1-score:  0.9167


### HW Unavailable

In [27]:
selected_dataset = "hw_unavailable"
log_lines = read_log_file(selected_dataset, log_datasets)

messages = build_messages(log_lines, user_anomaly_definition[selected_dataset])

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [28]:
print(response)

(1) normal
(2) abnormal - Component 'alt0' is in the unavailable state
(3) abnormal - Component 'alt0' is in the unavailable state
(4) abnormal - Component 'alt0' is in the unavailable state
(5) abnormal - Component 'alt0' is in the unavailable state
(6) abnormal - Component 'alt0' is in the unavailable state
(7) normal
(8) normal
(9) normal
(10) normal
(11) normal
(12) normal
(13) normal
(14) normal
(15) abnormal - Component 'alt0' is in the unavailable state
(16) abnormal - Component 'alt0' is in the unavailable state
(17) abnormal - Component 'alt0' is in the unavailable state
(18) abnormal - Component 'alt0' is in the unavailable state
(19) abnormal - Component 'alt0' is in the unavailable state
(20) normal
(21) normal
(22) normal
(23) normal
(24) normal
(25) normal


In [29]:
evaluate_predictions(response, selected_dataset, labels_datasets)

=== Evaluation hw_unavailable Results ===
Accuracy:  0.9600
Precision: 1.0000
Recall:    0.9091
F1-score:  0.9524
